# WARNING - Access GES DISC Data Using Python
Please, be very judicious when working on long data time series residing on a remote data server.

It is very likely that attempts to apply similar approaches on remote data, such as hourly data, for more than a year of data at a time, will result in a heavy load on the remote data server. This may lead to negative consequences, ranging from very slow performance that will be experienced by hundreds of other users, up to denial of service.

# Setup

## Library Imports

In [6]:
import requests
import os
import time
import platform
import warnings
import re
warnings.filterwarnings("ignore")
import dask

import xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import netCDF4

from calendar import monthrange
from urllib.parse import urlparse


## Helper Functions

In [8]:
def get_filename_from_cd(cd):
    """
    Get filename from content-disposition
    """
    if not cd:
        return None
    fname = re.findall('filename="?(.+)"?', cd)
    if len(fname) == 0:
        return None
    return fname[0].strip('"')

def ensure_directory_exists(folder):
    """Ensure the specified folder exists, create it if it doesn't."""
    if not os.path.exists(folder):
        os.makedirs(folder)
        print(f'{folder} did not exist. Creating...')
        

## Set data directory where files will be saved

In [26]:
# Set relative path from script
save_directory = '../DATA/M2T1NXAER/raw_data/'
output_save_directory = '../DATA/M2T1NXAER/processed/'

# Convert to absolute path
data_output_path = os.path.abspath(save_directory)

project_data_output_path = os.path.abspath(output_save_directory)

# Ensure the save directory exists
ensure_directory_exists(data_output_path)

ensure_directory_exists(project_data_output_path)

print(f'The raw data will be saved in {data_output_path}')
print(f'The processed files will be saved in {project_data_output_path}')

The raw data will be saved in C:\nextcloud\steglitz\GIS\IAAEU\pollution-data\DATA\M2T1NXAER\raw_data
The processed files will be saved in C:\nextcloud\steglitz\GIS\IAAEU\pollution-data\DATA\M2T1NXAER\processed


# Download Data

## Login Prompt to access NASA Earth Observational Data 

In [3]:
# This only needs to be run once. 

from subprocess import Popen
from getpass import getpass
import platform
import shutil

print("platform.python_version() ", platform.python_version())

urs = 'urs.earthdata.nasa.gov'    # Earthdata URL to call for authentication
prompts = ['\n Enter NASA Earthdata Login Username \n(or create an account at urs.earthdata.nasa.gov): ',
           '\n Enter NASA Earthdata Login Password: ', '\n" "']

homeDir = os.path.expanduser("~") + os.sep

with open(homeDir + '.netrc', 'w') as file:
    file.write('machine {} login {} password {}'.format(urs, getpass(prompt=prompts[0]), getpass(prompt=prompts[1])))
    file.close()
with open(homeDir + '.urs_cookies', 'w') as file:
    file.write('')
    file.close()
with open(homeDir + '.dodsrc', 'w') as file:
    file.write('HTTP.COOKIEJAR={}.urs_cookies\n'.format(homeDir))
    file.write('HTTP.NETRC={}.netrc'.format(homeDir))
    file.close()

print('Saved .netrc, .urs_cookies, and .dodsrc to:', homeDir)

# Set appropriate permissions for Linux/macOS
if platform.system() != "Windows":
    Popen('chmod og-rw ~/.netrc', shell=True)
else:
    # Copy dodsrc to working directory in Windows  
    shutil.copy2(homeDir + '.dodsrc', os.getcwd())
    print('Copied .dodsrc to:', os.getcwd())

platform.python_version()  3.12.7



 Enter NASA Earthdata Login Username 
(or create an account at urs.earthdata.nasa.gov):  ········

 Enter NASA Earthdata Login Password:  ········


Saved .netrc, .urs_cookies, and .dodsrc to: C:\Users\steglitz\
Copied .dodsrc to: C:\nextcloud\steglitz\GIS\IAAEU\pollution-data\notebooks


In [7]:
# Add a User-Agent
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/58.0.3029.110 Safari/537.3'}


## Set Download URLs

Either set the URL string to a specific URL or a txt file with the list of URLs that NASA provides after configuring your data request.

In [12]:

#url_file = 'subset_M2TMNXAER_5.12.4_20240112_094458_.txt'
url_file = 'subset_M2TMNXAER_5.12.4_20260305_093255.txt'

## Verify Link list (Optional)

In [15]:
# Set number of lines to print
number_of_lines_to_print = 5

# Open the file
with open(url_file, 'r') as file:
    # Read and print the first few lines
    for _ in range(number_of_lines_to_print):
        line = file.readline().strip()
        if not line:
            break
        # Print each line; end='' prevents adding extra newlines    
        print(f'The raw link is {repr(line)}\n', end='') 

The raw link is 'https://goldsmr4.gesdisc.eosdis.nasa.gov/data/MERRA2_MONTHLY/M2TMNXAER.5.12.4/doc/MERRA2.README.pdf'
The raw link is 'https://data.gesdisc.earthdata.nasa.gov/data/MERRA2_MONTHLY/M2TMNXAER.5.12.4/1980/MERRA2_100.tavgM_2d_aer_Nx.198001.nc4'


## Store URLs list

Removes the PDFs and stores the rest of the links in a list for processing.

In [16]:
#Initialize empty url list.
urls = []

with open(url_file, 'r') as file:
    for line in file:
        # Strip newlines
        url = line.strip()
        # Append if the file is not a pdf.
        if not url.lower().endswith('.pdf'):
            urls.append(url)

## Count downloadable URLs (Optional)

This is helpful if you have a long list of links.

In [17]:
%%time

# Get the number of files
def count_lines(urls):
    return len(urls)

print(f"There are {count_lines(urls)} links in the file.")

There are 1 links in the file.
CPU times: total: 0 ns
Wall time: 0 ns


## Download Data

**Warning** This can be very time consuming! Only run once to get the initial data. Running it again takes about as much time as each link is checked.

In [18]:
# Loop through each URL
for l in urls:
    url= l.strip()

    # Skip PDF files
    if url.lower().endswith('.pdf'):
        print(f'URL is a PDF, skipping: {url}')
        continue

    try:
        response = requests.get(url, stream=True)
        response.raise_for_status()

        # Attempt to get the filename from Content-Disposition header
        filename = get_filename_from_cd(response.headers.get('content-disposition'))
        if not filename:
            # Fallback to extracting filename from URL
            parsed_url = urlparse(url)
            filename = os.path.basename(parsed_url.path)

        full_path = os.path.join(data_output_path, filename)

        # Check if the file already exists
        if os.path.exists(full_path):
            print(f'File already exists, skipping download: {full_path}')
            continue

        # Write the file content
        with open(full_path, 'wb') as f:
            for chunk in response.iter_content(chunk_size=8192):
                f.write(chunk)

        print(f'File downloaded and saved to {full_path}')

    except Exception as e:
        print(f'Error with URL {url}: {e}')

File downloaded and saved to C:\nextcloud\steglitz\GIS\IAAEU\pollution-data\DATA\M2T1NXAER\raw_data\MERRA2_100.tavgM_2d_aer_Nx.198001.nc4


## Get list of downloaded nc4 files

In [19]:
import glob

nc4_files = glob.glob(data_output_path + '/*.nc4')

# Get list of files from a non-standard directory
# nc4_files = glob.glob(r'C:/Users/steglitz/School/Uni Trier/HIWI/IAAEU/project_files/AirQuality/data/M2T1NXAER/raw_data/1980_1989/' + '*.nc4')

In [20]:
# List nc4 files
nc4_files

['C:\\nextcloud\\steglitz\\GIS\\IAAEU\\pollution-data\\DATA\\M2T1NXAER\\raw_data\\MERRA2_100.tavgM_2d_aer_Nx.198001.nc4']

In [21]:
print(f'Number of files found: {len(nc4_files)}')

Number of files found: 1


In [22]:
# Used if you have multiple files that represent a single dataset.

#nc = netCDF4.MFDataset(nc4_files)

In [23]:
# from xarray.core.parallelcompat import list_chunkmanagers

In [24]:
# list_chunkmanagers()

## Data Pre-Processing

### Create single nc4 timeseries dataset

This combines multiple nc4 files into a single data set that can be manipulated into different time groupings.

In [27]:
ds = xr.open_mfdataset(nc4_files, combine='nested', concat_dim = 'time', parallel=True)
   
# View metadata (function like ncdump -c)
#ds

### Save data to single netcdf file

Combine all files into a single nc4 file

In [28]:
ds.to_netcdf(project_data_output_path + '/Pollution_1980.nc4')

# Start here to use already downloaded and prepared nc4 files

Start here to pre-process files that have already been downloaded. 

## Read in nc4 file

In [34]:
# Use full process
#nc4_all = netCDF4.Dataset(project_data_output_path + '/Pollution_1994.nc4')

file = project_data_output_path + '/Pollution_1980.nc4'

# Go directly to a file
#nc4_all = netCDF4.Dataset(file, 'r')

In [35]:
# Load data to xarray
ds=xr.open_dataset(file)

In [36]:
print(f'Variables include {nc4_all.variables.keys()}')

Variables include dict_keys(['BCANGSTR', 'BCCMASS', 'BCEXTTAU', 'BCFLUXU', 'BCFLUXV', 'BCSCATAU', 'BCSMASS', 'DMSCMASS', 'DMSSMASS', 'DUANGSTR', 'DUCMASS', 'DUCMASS25', 'DUEXTT25', 'DUEXTTAU', 'DUFLUXU', 'DUFLUXV', 'DUSCAT25', 'DUSCATAU', 'DUSMASS', 'DUSMASS25', 'OCANGSTR', 'OCCMASS', 'OCEXTTAU', 'OCFLUXU', 'OCFLUXV', 'OCSCATAU', 'OCSMASS', 'SO2CMASS', 'SO2SMASS', 'SO4CMASS', 'SO4SMASS', 'SSANGSTR', 'SSCMASS', 'SSCMASS25', 'SSEXTT25', 'SSEXTTAU', 'SSFLUXU', 'SSFLUXV', 'SSSCAT25', 'SSSCATAU', 'SSSMASS', 'SSSMASS25', 'SUANGSTR', 'SUEXTTAU', 'SUFLUXU', 'SUFLUXV', 'SUSCATAU', 'TOTANGSTR', 'TOTEXTTAU', 'TOTSCATAU', 'Var_BCANGSTR', 'Var_BCCMASS', 'Var_BCEXTTAU', 'Var_BCFLUXU', 'Var_BCFLUXV', 'Var_BCSCATAU', 'Var_BCSMASS', 'Var_DMSCMASS', 'Var_DMSSMASS', 'Var_DUANGSTR', 'Var_DUCMASS', 'Var_DUCMASS25', 'Var_DUEXTT25', 'Var_DUEXTTAU', 'Var_DUFLUXU', 'Var_DUFLUXV', 'Var_DUSCAT25', 'Var_DUSCATAU', 'Var_DUSMASS', 'Var_DUSMASS25', 'Var_OCANGSTR', 'Var_OCCMASS', 'Var_OCEXTTAU', 'Var_OCFLUXU', 'Var_O

In [37]:
# Print dimensions for each data item #

for var_name, variable in nc4_all.variables.items():
    print(f'{var_name}: {variable.dimensions}')

BCANGSTR: ('time', 'lat', 'lon')
BCCMASS: ('time', 'lat', 'lon')
BCEXTTAU: ('time', 'lat', 'lon')
BCFLUXU: ('time', 'lat', 'lon')
BCFLUXV: ('time', 'lat', 'lon')
BCSCATAU: ('time', 'lat', 'lon')
BCSMASS: ('time', 'lat', 'lon')
DMSCMASS: ('time', 'lat', 'lon')
DMSSMASS: ('time', 'lat', 'lon')
DUANGSTR: ('time', 'lat', 'lon')
DUCMASS: ('time', 'lat', 'lon')
DUCMASS25: ('time', 'lat', 'lon')
DUEXTT25: ('time', 'lat', 'lon')
DUEXTTAU: ('time', 'lat', 'lon')
DUFLUXU: ('time', 'lat', 'lon')
DUFLUXV: ('time', 'lat', 'lon')
DUSCAT25: ('time', 'lat', 'lon')
DUSCATAU: ('time', 'lat', 'lon')
DUSMASS: ('time', 'lat', 'lon')
DUSMASS25: ('time', 'lat', 'lon')
OCANGSTR: ('time', 'lat', 'lon')
OCCMASS: ('time', 'lat', 'lon')
OCEXTTAU: ('time', 'lat', 'lon')
OCFLUXU: ('time', 'lat', 'lon')
OCFLUXV: ('time', 'lat', 'lon')
OCSCATAU: ('time', 'lat', 'lon')
OCSMASS: ('time', 'lat', 'lon')
SO2CMASS: ('time', 'lat', 'lon')
SO2SMASS: ('time', 'lat', 'lon')
SO4CMASS: ('time', 'lat', 'lon')
SO4SMASS: ('time', '

# Process Data

## Resample Data

Resample hourly files into daily, weekly, and monthly files and calculate their corresponding statistics (e.g., mean, sum, maximum, and minimum).  

In [38]:
# Functions used to calculate various statistics
# ===================================
# Purpose           Function
# 
# mean of dim       mean
# sum of dim        sum
# max of dim        max
# min of dim        min
# ==================================

## Select Relevant Variable

In [39]:
sel_var_shortname = "DUSMASS25"
sel_var_value= ds[sel_var_shortname]
sel_var_longname = sel_var_value.attrs['long_name']
sel_var_unit = '('+sel_var_value.attrs['units']+')' 

print("The selected variable is {}: {}{}".format(sel_var_shortname, sel_var_longname,sel_var_unit))

The selected variable is DUSMASS25: Dust Surface Mass Concentration - PM 2.5(kg m-3)


## Daily Statistics

### Daily Mean

In [40]:
%time

sel_var_daily_mean = sel_var_value.resample(time="1D").mean(dim='time', skipna=True)
sel_var_daily_mean

CPU times: total: 0 ns
Wall time: 0 ns


<xarray.DataArray 'DUSMASS25' (time: 1, lat: 361, lon: 576)> Size: 832kB
array([[[4.8421881e-11, 4.8421881e-11, 4.8421881e-11, ...,
         4.8421881e-11, 4.8421881e-11, 4.8421881e-11],
        [4.8432643e-11, 4.8432463e-11, 4.8432463e-11, ...,
         4.8434000e-11, 4.8434000e-11, 4.8434000e-11],
        [4.9018775e-11, 4.9018775e-11, 4.9018775e-11, ...,
         4.9018775e-11, 4.9018775e-11, 4.9018775e-11],
        ...,
        [3.8356451e-10, 3.8356451e-10, 3.8356451e-10, ...,
         3.8356451e-10, 3.8356451e-10, 3.8356451e-10],
        [3.9362610e-10, 3.9400161e-10, 3.9400161e-10, ...,
         3.9279913e-10, 3.9279913e-10, 3.9279913e-10],
        [3.9968398e-10, 3.9968398e-10, 3.9968398e-10, ...,
         3.9968398e-10, 3.9968398e-10, 3.9968398e-10]]],
      shape=(1, 361, 576), dtype=float32)
Coordinates:
  * time     (time) datetime64[ns] 8B 1980-01-01
  * lat      (lat) float64 3kB -90.0 -89.5 -89.0 -88.5 ... 88.5 89.0 89.5 90.0
  * lon      (lon) float64 5kB -180.0 -179.4 -178.8 -178.1 ... 178.1 178.8 179.4
Attributes:
    long_name:       Dust Surface Mass Concentration - PM 2.5
    units:           kg m-3
    fmissing_value:  1e+15
    vmax:            1e+15
    vmin:            -1e+15
    valid_range:     [-1.e+15  1.e+15]

### Add CRS to netCDF output

In [42]:
sel_var_daily_mean.attrs['crs']= 'EPSG:4326'

### Create time lables for each day.

In [43]:
daily_labels = sel_var_daily_mean['time'].dt.day.values

### Rename bands with day names.

In [44]:
sel_var_daily_mean = sel_var_daily_mean.assign_coords({'time': daily_labels})

### Save to netCDF file

In [45]:
sel_var_daily_mean.to_netcdf(project_data_output_path + '/Pollution_1980_daily_mean_PM25.nc4')

## Weekly Statistics

## Weekly Mean

In [46]:
%%time

sel_var_weekly_mean = sel_var_value.resample(time='1w').mean(dim='time', skipna=True)
#sel_var_weekly_mean

CPU times: total: 0 ns
Wall time: 13.4 ms


### Add CRS to netCDF output

In [47]:
sel_var_weekly_mean.attrs['crs']= 'EPSG:4326'

### Create time lables for each week.

In [48]:
weekly_labels = sel_var_weekly_mean['time'].dt.week.values

### Rename bands with week names.

In [49]:
sel_var_weekly_mean = sel_var_weekly_mean.assign_coords({'time': weekly_labels})

### Save to netCDF file

In [50]:
sel_var_weekly_mean.to_netcdf(project_data_output_path + '/Pollution_1980_weekly_mean_PM25.nc4')

## Monthly Statistics

### Monthly Mean

In [60]:
%%time
sel_var_monthly_mean = sel_var_value.resample(time='m').mean(dim='time', skipna=True)
#sel_var_monthly_mean

CPU times: total: 15.6 ms
Wall time: 8 ms


### Add CRS to netCDF output

In [61]:
sel_var_monthly_mean.attrs['crs']= 'EPSG:4326'

### Create time lables for each month.

In [62]:
month_labels = sel_var_monthly_mean['time'].dt.month.values

### Rename bands with month names.

In [63]:
sel_var_monthly_mean = sel_var_monthly_mean.assign_coords({'time': month_labels})

### Save data to netCDF file

In [64]:
sel_var_monthly_mean.to_netcdf(project_data_output_path + '/Pollution_1990_monthly_mean_PM25.nc4')

## Yearly Statistics

### Yearly Mean

In [65]:
%%time

sel_var_yearly_mean = sel_var_value.resample(time='Y').mean(dim='time', skipna=True)
#sel_var_yearly_mean = sel_var_value.resample(time='Y').mean()
#sel_var_yearly_mean

CPU times: total: 15.6 ms
Wall time: 8.4 ms


### Create time lables for each year.

In [66]:
year_labels = sel_var_yearly_mean['time'].dt.year.values

### Rename bands with year names.

In [68]:
sel_var_yearly_mean = sel_var_yearly_mean.assign_coords({'time': year_labels})

### Add CRS to netCDF output

In [69]:
sel_var_yearly_mean.attrs['crs']= 'EPSG:4326'

### Save data to netcdf file

In [70]:
sel_var_yearly_mean.to_netcdf(project_data_output_path + '/Pollution_1980_yearly_mean_PM25.nc4', engine='netcdf4')

## Total Time-period Mean #

### Calculate average across the entire time.

In [71]:
sel_var_10_year_mean = sel_var_value.mean(dim='time', skipna=True)
#sel_var_10_year_mean = sel_var_value.resample(time='10Y').mean(dim='time', skipna=True)

### Add CRS to netCDF output

In [72]:
sel_var_10_year_mean.attrs['crs']= 'EPSG:4326'

### Save data to netcdf file

In [73]:
sel_var_10_year_mean.to_netcdf(project_data_output_path + '/Pollution_1998_10-year_mean_PM25.nc4')